# Final Generalized NOM with Proper Augmented Lagrangian

This version preserves the original NOM workflow but improves constraint handling.

### Preserved NOM logic
1. Learn the linear objective using a neural-network surrogate.
2. Convert all constraints to `A x <= b`.
3. For LSIP, discretize the continuous parameter `t`.
4. Generate the decision variables through a trainable `start_layer`.
5. Optimize with Adam from multiple starting points.

### Added improvements
- Proper PHR inequality augmented Lagrangian.
- One Lagrange multiplier per constraint.
- Adaptive penalty parameter `rho`.
- Primal-feasibility, complementarity, stationarity and KKT diagnostics.
- Gradient clipping.
- Best-feasible-iterate tracking.
- Dense validation of the original semi-infinite constraint.
- Independent SciPy/HiGHS LP benchmark for correctness checking.
- CSV history of augmented-Lagrangian outer iterations.

## 1. Load the complete implementation

Run this cell first. It defines every helper, model class and optimizer.

In [1]:

"""
Interactive Generic NOM
=======================

This is a generalized version of the original NOM notebook.

Core NOM logic intentionally preserved:
1. Learn the linear objective with a neural-network surrogate.
2. Convert all constraints to A @ x <= b.
3. For a semi-infinite problem, discretize t over a finite grid.
4. Generate decision variables through a trainable start_layer.
5. Minimize: learned objective + constraint-violation penalty.
6. Use Adam and multiple starting points.
7. Support the original standard penalty and quadratic-augmented penalty.

The main change is that the problem definition is now entered by the user,
so the dimensions and coefficients are no longer hard-coded to 3 variables.
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.distributions.uniform as urand
from scipy.optimize import linprog


# ============================================================
# 0. REPRODUCIBILITY + DEVICE
# ============================================================

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 1. SMALL INPUT HELPERS
# ============================================================

def _read_int(prompt, minimum=1):
    while True:
        try:
            value = int(input(prompt).strip())
            if value < minimum:
                print(f"Please enter an integer >= {minimum}.")
                continue
            return value
        except ValueError:
            print("Please enter a valid integer.")


def _read_float(prompt):
    while True:
        try:
            return float(input(prompt).strip())
        except ValueError:
            print("Please enter a valid number.")


def _read_yes_no(prompt, default=True):
    suffix = " [Y/n]: " if default else " [y/N]: "
    while True:
        ans = input(prompt + suffix).strip().lower()
        if ans == "":
            return default
        if ans in {"y", "yes"}:
            return True
        if ans in {"n", "no"}:
            return False
        print("Please answer y or n.")


def _read_relation(prompt="Relation (<=, >=, =): "):
    aliases = {
        "<": "<=", "<=": "<=",
        ">": ">=", ">=": ">=",
        "=": "=", "==": "=",
    }
    while True:
        raw = input(prompt).strip()
        if raw in aliases:
            return aliases[raw]
        print("Please enter <=, >=, or =.")


def _read_vector(prompt, n):
    while True:
        raw = input(prompt).strip().replace(",", " ")
        try:
            values = [float(v) for v in raw.split()]
        except ValueError:
            print("Please enter only numeric coefficients.")
            continue

        if len(values) != n:
            print(f"Expected exactly {n} coefficients, but received {len(values)}.")
            continue
        return np.asarray(values, dtype=np.float64)


def _normalize_problem_type(raw):
    raw = raw.strip().lower().replace("_", " ").replace("-", " ")
    if raw in {"1", "linear", "lp", "finite", "finite linear"}:
        return "linear"
    if raw in {
        "2", "semi", "semi linear", "semilinear", "lsip",
        "semi infinite", "linear semi infinite",
        "linear semi infinite programming",
    }:
        return "semi_infinite"
    raise ValueError("Choose 1/linear or 2/semi-linear (LSIP).")


# ============================================================
# 2. t-EXPRESSION PARSER FOR SEMI-INFINITE CONSTRAINTS
# ============================================================

def evaluate_t_expression(expr, t):
    """
    Evaluate a coefficient/RHS expression over a torch tensor t.

    Examples:
        1
        t
        t + t**2
        exp(t)
        0.5 + t
        sin(t)
        sqrt(1+t)

    The problem remains LINEAR IN x. Expressions may be nonlinear only in t.
    """
    expr = expr.strip().replace("^", "**")

    allowed = {
        "t": t,
        "exp": torch.exp,
        "sin": torch.sin,
        "cos": torch.cos,
        "tan": torch.tan,
        "sqrt": torch.sqrt,
        "log": torch.log,
        "abs": torch.abs,
        "pi": float(np.pi),
        "e": float(np.e),
    }

    try:
        value = eval(expr, {"__builtins__": {}}, allowed)
    except Exception as exc:
        raise ValueError(
            f"Could not evaluate expression '{expr}'. "
            "Use forms such as 1, t, t**2, t+t**2, exp(t), sin(t)."
        ) from exc

    if isinstance(value, (int, float, np.number)):
        return torch.full_like(t, float(value))

    value = torch.as_tensor(value, dtype=t.dtype, device=t.device)
    if value.ndim == 0:
        value = torch.full_like(t, float(value.item()))

    if value.shape != t.shape:
        raise ValueError(
            f"Expression '{expr}' produced shape {tuple(value.shape)}, "
            f"but expected {tuple(t.shape)}."
        )
    return value


# ============================================================
# 3. HUMAN-INPUT PROBLEM DEFINITION
# ============================================================

def collect_problem_from_user():
    print("\n" + "=" * 72)
    print("INTERACTIVE NEURAL OPTIMIZATION MACHINE (NOM)")
    print("=" * 72)
    print("Choose the type of problem:")
    print("  1. Linear programming (finite linear constraints)")
    print("  2. Semi-linear / Linear Semi-Infinite Programming (LSIP)")
    print("     Finite x variables, but constraints indexed by t in an interval.")

    while True:
        try:
            problem_type = _normalize_problem_type(
                input("\nProblem type [1/2 or linear/semi]: ")
            )
            break
        except ValueError as exc:
            print(exc)

    n_vars = _read_int("\nHow many decision variables? ", minimum=1)

    while True:
        sense = input("Optimization sense [max/min]: ").strip().lower()
        if sense in {"max", "maximize", "maximise"}:
            sense = "max"
            break
        if sense in {"min", "minimize", "minimise"}:
            sense = "min"
            break
        print("Please enter max or min.")

    obj_coeffs = _read_vector(
        f"Enter the {n_vars} objective coefficients for x1..x{n_vars}, "
        "separated by spaces: ",
        n_vars,
    )

    nonnegative = _read_yes_no(
        "Are all decision variables constrained to be non-negative?",
        default=True,
    )

    problem = {
        "problem_type": problem_type,
        "n_vars": n_vars,
        "sense": sense,
        "objective_coeffs": obj_coeffs,
        "nonnegative": nonnegative,
        "finite_constraints": [],
        "semi_families": [],
    }

    if problem_type == "linear":
        # Allow zero ordinary constraints.
        while True:
            try:
                n_constraints = int(
                    input("\nHow many ordinary linear constraints? ").strip()
                )
                if n_constraints < 0:
                    raise ValueError
                break
            except ValueError:
                print("Please enter an integer >= 0.")

        for j in range(n_constraints):
            print(f"\nConstraint {j+1}")
            coeffs = _read_vector(
                f"  Enter {n_vars} coefficients: ",
                n_vars,
            )
            relation = _read_relation("  Relation (<=, >=, =): ")
            rhs = _read_float("  Right-hand side value: ")

            problem["finite_constraints"].append(
                {
                    "coeffs": coeffs,
                    "relation": relation,
                    "rhs": rhs,
                    "name": f"linear_constraint_{j+1}",
                }
            )

    else:
        print("\nSemi-infinite parameter interval")
        t_min = _read_float("  t minimum: ")
        t_max = _read_float("  t maximum: ")
        if not t_max > t_min:
            raise ValueError("t maximum must be greater than t minimum.")

        m = _read_int(
            "  Number of discretization points m "
            "(for example 11, 101, 1001): ",
            minimum=2,
        )

        n_families = _read_int(
            "\nHow many t-dependent semi-infinite constraint families? ",
            minimum=1,
        )

        problem["t_min"] = t_min
        problem["t_max"] = t_max
        problem["m"] = m

        print("\nAllowed t expressions include:")
        print("  1, t, t**2, t+t**2, exp(t), sin(t), cos(t), sqrt(1+t)")
        print("Each constraint must stay LINEAR in x; only its coefficients may depend on t.")

        for fam in range(n_families):
            print(f"\nSemi-infinite constraint family {fam+1}")
            coeff_exprs = []
            for i in range(n_vars):
                coeff_exprs.append(
                    input(f"  Coefficient of x{i+1} as a function of t: ").strip()
                )

            relation = _read_relation(
                "  Relation that must hold for every t (<=, >=, =): "
            )
            rhs_expr = input("  Right-hand side as a function of t: ").strip()

            problem["semi_families"].append(
                {
                    "coeff_exprs": coeff_exprs,
                    "relation": relation,
                    "rhs_expr": rhs_expr,
                    "name": f"semi_family_{fam+1}",
                }
            )

        # Optional ordinary constraints can coexist with an LSIP.
        while True:
            try:
                n_fixed = int(
                    input(
                        "\nHow many additional ordinary linear constraints? "
                        "[enter 0 if none]: "
                    ).strip()
                )
                if n_fixed < 0:
                    raise ValueError
                break
            except ValueError:
                print("Please enter an integer >= 0.")

        for j in range(n_fixed):
            print(f"\nAdditional fixed constraint {j+1}")
            coeffs = _read_vector(
                f"  Enter {n_vars} coefficients: ",
                n_vars,
            )
            relation = _read_relation("  Relation (<=, >=, =): ")
            rhs = _read_float("  Right-hand side value: ")
            problem["finite_constraints"].append(
                {
                    "coeffs": coeffs,
                    "relation": relation,
                    "rhs": rhs,
                    "name": f"fixed_constraint_{j+1}",
                }
            )

    print("\nNeural surrogate training interval")
    train_low = _read_float(
        "  Lower value used to generate training samples "
        "(e.g. 0 for nonnegative problems): "
    )
    train_high = _read_float("  Upper value used to generate training samples: ")
    if not train_high > train_low:
        raise ValueError("Training upper bound must be greater than lower bound.")

    problem["train_interval"] = (train_low, train_high)

    print("\nConstraint-handling method:")
    print("  1. Standard penalty: rho * sum(violation)")
    print("  2. Quadratic-augmented penalty: rho * sum(v + 0.5*v^2)")
    print("  3. Proper inequality Augmented Lagrangian (recommended)")
    print("     Uses one multiplier per constraint, adaptive rho, and KKT checks.")
    while True:
        penalty_choice = input("Choose 1, 2, or 3 [default 3]: ").strip()
        if penalty_choice in {"", "3"}:
            problem["penalty_mode"] = "phr_augmented_lagrangian"
            break
        if penalty_choice == "2":
            problem["penalty_mode"] = "augmented"
            break
        if penalty_choice == "1":
            problem["penalty_mode"] = "standard"
            break
        print("Please enter 1, 2, or 3.")

    return problem


# ============================================================
# 4. CONVERT USER PROBLEM TO A*x <= b
# ============================================================

def _append_relation(rows_A, rows_b, A_piece, b_piece, relation):
    """
    Convert <=, >=, = into one or two <= systems without changing the NOM logic.
    """
    A_piece = torch.as_tensor(A_piece, dtype=torch.float32)
    b_piece = torch.as_tensor(b_piece, dtype=torch.float32)

    if A_piece.ndim == 1:
        A_piece = A_piece.unsqueeze(0)
    if b_piece.ndim == 0:
        b_piece = b_piece.unsqueeze(0)

    if relation == "<=":
        rows_A.append(A_piece)
        rows_b.append(b_piece)
    elif relation == ">=":
        rows_A.append(-A_piece)
        rows_b.append(-b_piece)
    elif relation == "=":
        rows_A.append(A_piece)
        rows_b.append(b_piece)
        rows_A.append(-A_piece)
        rows_b.append(-b_piece)
    else:
        raise ValueError(f"Unknown relation: {relation}")


def build_constraint_matrix(problem):
    n = problem["n_vars"]
    rows_A = []
    rows_b = []
    metadata = []

    # Semi-infinite constraints -> ordinary discretization.
    if problem["problem_type"] == "semi_infinite":
        t = torch.linspace(
            float(problem["t_min"]),
            float(problem["t_max"]),
            int(problem["m"]),
            dtype=torch.float32,
        )

        for family in problem["semi_families"]:
            cols = [
                evaluate_t_expression(expr, t)
                for expr in family["coeff_exprs"]
            ]
            A_family = torch.column_stack(cols)
            b_family = evaluate_t_expression(family["rhs_expr"], t)

            before = sum(x.shape[0] for x in rows_A)
            _append_relation(
                rows_A,
                rows_b,
                A_family,
                b_family,
                family["relation"],
            )
            after = sum(x.shape[0] for x in rows_A)
            metadata.append(
                {
                    "name": family["name"],
                    "type": "semi_infinite_discretized",
                    "rows_added": after - before,
                }
            )

    # Ordinary finite linear constraints.
    for c in problem["finite_constraints"]:
        A_piece = torch.tensor(c["coeffs"], dtype=torch.float32)
        b_piece = torch.tensor(float(c["rhs"]), dtype=torch.float32)

        before = sum(x.shape[0] for x in rows_A)
        _append_relation(
            rows_A, rows_b, A_piece, b_piece, c["relation"]
        )
        after = sum(x.shape[0] for x in rows_A)
        metadata.append(
            {
                "name": c["name"],
                "type": "finite_linear",
                "rows_added": after - before,
            }
        )

    # Original notebook logic: x >= 0 -> -I x <= 0.
    if problem["nonnegative"]:
        A_nonnegative = -torch.eye(n, dtype=torch.float32)
        b_nonnegative = torch.zeros(n, dtype=torch.float32)
        rows_A.append(A_nonnegative)
        rows_b.append(b_nonnegative)
        metadata.append(
            {
                "name": "nonnegativity",
                "type": "bounds",
                "rows_added": n,
            }
        )

    if not rows_A:
        # NOM can technically work without explicit constraints.
        A = torch.empty((0, n), dtype=torch.float32)
        b = torch.empty((0,), dtype=torch.float32)
    else:
        A = torch.cat(rows_A, dim=0)
        b = torch.cat(rows_b, dim=0)

    if A.shape[1] != n:
        raise RuntimeError(
            f"Internal dimension error: A has {A.shape[1]} columns; expected {n}."
        )

    return A, b, metadata


# ============================================================
# 5. GENERIC OBJECTIVE DATASET
# ============================================================

class AlgebraicDataset(Dataset):
    """
    Same objective-learning idea as the original notebook,
    generalized from 3 variables to n variables.
    """

    def __init__(self, objective_coeffs, sense, interval, nsamples):
        self.coeffs = torch.tensor(
            objective_coeffs,
            dtype=torch.float32,
        )

        n_vars = len(objective_coeffs)
        low, high = interval

        X = urand.Uniform(low, high).sample([nsamples, n_vars])
        original_y = X @ self.coeffs

        # Adam minimizes, so max(c^T x) becomes min(-c^T x).
        y = -original_y if sense == "max" else original_y

        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train(model, dataloader, lossfunc, optimizer):
    model.train()
    cumloss = 0.0

    for X, y in dataloader:
        X = X.float().to(device)
        y = y.unsqueeze(1).float().to(device)

        pred = model(X)
        loss = lossfunc(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cumloss += loss.item()

    return cumloss / max(1, len(dataloader))


def test(model, dataloader, lossfunc):
    model.eval()
    cumloss = 0.0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.float().to(device)
            y = y.unsqueeze(1).float().to(device)

            pred = model(X)
            loss = lossfunc(pred, y)
            cumloss += loss.item()

    return cumloss / max(1, len(dataloader))


def train_objective_surrogate(
    problem,
    train_nsamples=10_000,
    test_nsamples=1_000,
    batch_size=64,
    epochs=1_000,
    learning_rate=0.002,
    verbose=True,
):
    """
    Preserve the original objective-learning architecture:
        n -> 64 -> ReLU -> 32 -> ReLU -> 1
    """
    train_dataset = AlgebraicDataset(
        problem["objective_coeffs"],
        problem["sense"],
        problem["train_interval"],
        train_nsamples,
    )

    test_dataset = AlgebraicDataset(
        problem["objective_coeffs"],
        problem["sense"],
        problem["train_interval"],
        test_nsamples,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    n = problem["n_vars"]
    model = nn.Sequential(
        nn.Linear(n, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
    ).to(device)

    lossfunc = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    if verbose:
        print("\nTraining the objective surrogate...")

    for epoch in range(epochs):
        train_loss = train(model, train_loader, lossfunc, optimizer)

        if verbose and (epoch == 0 or epoch % 500 == 0 or epoch == epochs - 1):
            val_loss = test(model, test_loader, lossfunc)
            print(
                f"  epoch {epoch:4d} | "
                f"train MSE={train_loss:.8f} | "
                f"validation MSE={val_loss:.8f}"
            )

    final_test = test(model, test_loader, lossfunc)

    # Explicitly freeze weights after stage 1.
    # Gradients can still flow through the fixed network to its INPUT x.
    for p in model.parameters():
        p.requires_grad_(False)

    model.eval()

    if verbose:
        print(f"Final objective-surrogate test MSE: {final_test:.10f}")

    return model, final_test


# ============================================================
# 6. GENERIC NOM - CORE LOGIC PRESERVED
# ============================================================

class NeuralOptimizationMachine(nn.Module):
    """
    Standard penalty version from the original notebook:
        loss = objective_surrogate(x) + rho * sum(max(Ax-b, 0))
    """

    def __init__(self, obj_net, A, b, n_vars, penalty=20.0):
        super().__init__()
        self.obj_net = obj_net
        self.register_buffer(
            "A", torch.as_tensor(A, dtype=torch.float32)
        )
        self.register_buffer(
            "b", torch.as_tensor(b, dtype=torch.float32)
        )
        self.penalty = float(penalty)

        # Original NOM mechanism generalized from 3x3 to n x n.
        self.start_layer = nn.Linear(n_vars, n_vars)

    def forward(self, x0):
        x = self.start_layer(x0)
        obj_val = self.obj_net(x)

        if self.A.shape[0] == 0:
            penalty_sum = torch.zeros_like(obj_val)
        else:
            violation = torch.clamp(
                x @ self.A.T - self.b,
                min=0.0,
            )
            penalty_sum = (
                self.penalty
                * violation.sum(dim=1, keepdim=True)
            )

        nom_output = obj_val + penalty_sum
        return nom_output, x


class NeuralOptimizationMachineAugmented(nn.Module):
    """
    Quadratic-augmented penalty version from the original notebook:
        loss = objective_surrogate(x)
               + rho * sum(v + 0.5*v^2)
    where v = max(Ax-b, 0).
    """

    def __init__(self, obj_net, A, b, n_vars, penalty=20.0):
        super().__init__()
        self.obj_net = obj_net
        self.register_buffer(
            "A", torch.as_tensor(A, dtype=torch.float32)
        )
        self.register_buffer(
            "b", torch.as_tensor(b, dtype=torch.float32)
        )
        self.penalty = float(penalty)
        self.start_layer = nn.Linear(n_vars, n_vars)

    def forward(self, x0):
        x = self.start_layer(x0)
        obj_val = self.obj_net(x)

        if self.A.shape[0] == 0:
            penalty_sum = torch.zeros_like(obj_val)
        else:
            violation = torch.clamp(
                x @ self.A.T - self.b,
                min=0.0,
            )
            penalty_sum = self.penalty * (
                violation + 0.5 * violation**2
            ).sum(dim=1, keepdim=True)

        nom_output = obj_val + penalty_sum
        return nom_output, x


class NeuralOptimizationMachinePHR(nn.Module):
    """
    Proper Powell-Hestenes-Rockafellar (PHR) augmented Lagrangian
    for inequalities g(x) = A x - b <= 0.

    Loss:
        f_hat(x)
        + sum( (relu(lambda + rho*g)^2 - lambda^2) / (2*rho) )

    Multipliers are NOT optimized by Adam. They are updated after each
    inner optimization block:
        lambda <- relu(lambda + rho*g)

    This keeps the original NOM mechanism (a trainable start_layer produces x)
    while improving constraint handling near active boundaries.
    """

    def __init__(self, obj_net, A, b, n_vars, rho=1.0):
        super().__init__()
        self.obj_net = obj_net
        self.register_buffer(
            "A", torch.as_tensor(A, dtype=torch.float32)
        )
        self.register_buffer(
            "b", torch.as_tensor(b, dtype=torch.float32)
        )
        self.register_buffer(
            "lam", torch.zeros(len(b), dtype=torch.float32)
        )
        self.rho = float(rho)
        self.start_layer = nn.Linear(n_vars, n_vars)

    def reset_multipliers(self):
        with torch.no_grad():
            self.lam.zero_()

    def forward(self, x0):
        x = self.start_layer(x0)
        obj_val = self.obj_net(x)

        if self.A.shape[0] == 0:
            g = torch.empty(
                (x.shape[0], 0),
                dtype=x.dtype,
                device=x.device,
            )
            return obj_val, x, g

        g = x @ self.A.T - self.b

        shifted = torch.relu(
            self.lam.unsqueeze(0) + self.rho * g
        )

        al_term = (
            (
                shifted**2
                - self.lam.unsqueeze(0)**2
            )
            / (2.0 * self.rho)
        ).sum(dim=1, keepdim=True)

        loss = obj_val + al_term
        return loss, x, g

    @torch.no_grad()
    def update_multipliers(self, g):
        if self.A.shape[0] == 0:
            return
        g_vec = g.squeeze(0)
        self.lam.copy_(
            torch.relu(self.lam + self.rho * g_vec)
        )


# ============================================================
# 7. NOM OPTIMIZATION
# ============================================================

def original_objective_value(x, problem):
    coeffs = np.asarray(
        problem["objective_coeffs"],
        dtype=np.float64,
    )
    return float(np.dot(coeffs, np.asarray(x, dtype=np.float64)))


def transformed_objective_value(x, problem):
    val = original_objective_value(x, problem)
    return -val if problem["sense"] == "max" else val


def max_constraint_violation_numpy(x, A, b):
    if len(b) == 0:
        return 0.0
    residual = A @ np.asarray(x, dtype=np.float64) - b
    return float(max(0.0, np.max(residual)))


def transformed_objective_coeffs(problem):
    """Exact coefficients of the minimization-form objective used for KKT checks."""
    c = np.asarray(problem["objective_coeffs"], dtype=np.float64)
    return -c if problem["sense"] == "max" else c


def kkt_metrics_numpy(problem, x, lam, A, b):
    """
    KKT diagnostics for the discretized problem written as

        min f(x)
        s.t. A x - b <= 0.

    We evaluate stationarity with the TRUE linear objective coefficients,
    not with the learned surrogate. This keeps the convergence test independent
    of surrogate approximation error.
    """
    x = np.asarray(x, dtype=np.float64)
    lam = np.asarray(lam, dtype=np.float64)
    A = np.asarray(A, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    if len(b) == 0:
        stationarity = float(np.max(np.abs(transformed_objective_coeffs(problem))))
        return {
            "primal_violation": 0.0,
            "complementarity": 0.0,
            "stationarity": stationarity,
            "kkt_merit": stationarity,
        }

    g = A @ x - b
    primal = float(max(0.0, np.max(g)))
    complementarity = float(np.max(np.abs(lam * g)))
    stationarity_vec = transformed_objective_coeffs(problem) + A.T @ lam
    stationarity = float(np.max(np.abs(stationarity_vec)))

    return {
        "primal_violation": primal,
        "complementarity": complementarity,
        "stationarity": stationarity,
        "kkt_merit": primal + complementarity + stationarity,
    }


def optimize_with_nom(
    problem,
    obj_model,
    A,
    b,
    penalty=20.0,
    n_starting_points=5,
    n_epochs_per_start=10_000,
    learning_rate=0.001,
    feasibility_tol=1e-5,
    print_every=500,
    verbose=True,
):
    n = problem["n_vars"]

    if problem["penalty_mode"] == "standard":
        nom = NeuralOptimizationMachine(
            obj_model, A, b, n, penalty=penalty
        ).to(device)
    else:
        nom = NeuralOptimizationMachineAugmented(
            obj_model, A, b, n, penalty=penalty
        ).to(device)

    low, high = problem["train_interval"]

    torch.manual_seed(SEED)
    starting_points = (
        torch.rand(n_starting_points, n)
        * (high - low)
        + low
    )

    A_np = A.detach().cpu().numpy().astype(np.float64)
    b_np = b.detach().cpu().numpy().astype(np.float64)

    all_start_rows = []

    # Monitoring improvement only:
    # retain best FEASIBLE iterate while preserving the same NOM loss/updates.
    best_feasible = None
    best_any = None

    started = time.perf_counter()

    for start_idx, x0 in enumerate(starting_points):
        x0 = x0.to(device).unsqueeze(0)

        # Original logic: each start receives a fresh trainable start layer.
        nom.start_layer.reset_parameters()
        optimizer = torch.optim.Adam(
            nom.start_layer.parameters(),
            lr=learning_rate,
        )

        best_this_start = None

        if verbose:
            print(
                f"\n--- Starting point {start_idx}: "
                f"{x0.squeeze().detach().cpu().numpy()} ---"
            )

        for epoch in range(n_epochs_per_start):
            optimizer.zero_grad()
            loss, x_current = nom(x0)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                # Recompute AFTER optimizer.step() for accurate monitoring.
                loss_eval, x_eval = nom(x0)
                x_np = (
                    x_eval.squeeze(0)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )

                obj_original = original_objective_value(
                    x_np, problem
                )
                obj_transformed = transformed_objective_value(
                    x_np, problem
                )
                max_violation = max_constraint_violation_numpy(
                    x_np, A_np, b_np
                )

                candidate = {
                    "start": start_idx,
                    "epoch": epoch,
                    "x": x_np.copy(),
                    "objective_original": obj_original,
                    "objective_transformed": obj_transformed,
                    "max_violation": max_violation,
                    "nom_loss": float(loss_eval.item()),
                }

                if (
                    best_any is None
                    or candidate["nom_loss"] < best_any["nom_loss"]
                ):
                    best_any = candidate.copy()

                if max_violation <= feasibility_tol:
                    if (
                        best_this_start is None
                        or obj_transformed
                        < best_this_start["objective_transformed"]
                    ):
                        best_this_start = candidate.copy()

                    if (
                        best_feasible is None
                        or obj_transformed
                        < best_feasible["objective_transformed"]
                    ):
                        best_feasible = candidate.copy()

            if verbose and (
                epoch == 0
                or epoch % print_every == 0
                or epoch == n_epochs_per_start - 1
            ):
                print(
                    f"  epoch {epoch:5d} | "
                    f"loss={candidate['nom_loss']:.6f} | "
                    f"original obj={candidate['objective_original']:.6f} | "
                    f"max violation={candidate['max_violation']:.3e} | "
                    f"x={candidate['x']}"
                )

        final_candidate = candidate.copy()

        chosen_for_start = (
            best_this_start
            if best_this_start is not None
            else final_candidate
        )

        row = {
            "start": start_idx,
            "best_epoch": chosen_for_start["epoch"],
            "objective_original": chosen_for_start["objective_original"],
            "objective_transformed": chosen_for_start["objective_transformed"],
            "max_violation": chosen_for_start["max_violation"],
            "nom_loss": chosen_for_start["nom_loss"],
            "feasible": bool(
                chosen_for_start["max_violation"] <= feasibility_tol
            ),
        }

        for j, value in enumerate(chosen_for_start["x"], start=1):
            row[f"x{j}"] = float(value)

        all_start_rows.append(row)

    runtime = time.perf_counter() - started

    selected = best_feasible if best_feasible is not None else best_any
    selected_feasible = (
        selected is not None
        and selected["max_violation"] <= feasibility_tol
    )

    summary = {
        "problem_type": problem["problem_type"],
        "sense": problem["sense"],
        "n_vars": n,
        "penalty_mode": problem["penalty_mode"],
        "penalty": float(penalty),
        "n_constraint_rows": int(A.shape[0]),
        "n_starting_points": int(n_starting_points),
        "epochs_per_start": int(n_epochs_per_start),
        "runtime_sec": float(runtime),
        "selected_start": int(selected["start"]),
        "selected_epoch": int(selected["epoch"]),
        "objective_original": float(selected["objective_original"]),
        "objective_transformed": float(selected["objective_transformed"]),
        "max_constraint_violation": float(selected["max_violation"]),
        "feasible": bool(selected_feasible),
    }

    for j, value in enumerate(selected["x"], start=1):
        summary[f"x{j}"] = float(value)

    return summary, pd.DataFrame(all_start_rows), selected


def solve_finite_lp_benchmark(problem, A, b):
    """
    Independent SciPy/HiGHS benchmark for the SAME discretized linear problem.
    It is used only for validation/reporting and never guides NOM updates.
    """
    c_min = transformed_objective_coeffs(problem)

    try:
        out = linprog(
            c=c_min,
            A_ub=A.detach().cpu().numpy().astype(np.float64),
            b_ub=b.detach().cpu().numpy().astype(np.float64),
            bounds=[(None, None)] * problem["n_vars"],
            method="highs",
        )
    except Exception as exc:
        return {
            "lp_success": False,
            "lp_message": str(exc),
        }

    if not out.success:
        return {
            "lp_success": False,
            "lp_message": out.message,
        }

    original_obj = original_objective_value(out.x, problem)

    result = {
        "lp_success": True,
        "lp_message": out.message,
        "lp_objective_original": float(original_obj),
        "lp_objective_transformed": float(out.fun),
    }
    for j, value in enumerate(out.x, start=1):
        result[f"lp_x{j}"] = float(value)

    return result


def optimize_with_nom_phr(
    problem,
    obj_model,
    A,
    b,
    rho0=1.0,
    rho_max=1000.0,
    rho_multiplier=2.0,
    violation_reduction_target=0.7,
    n_starting_points=5,
    outer_iterations=30,
    inner_epochs=300,
    learning_rate=0.001,
    feasibility_tol=1e-5,
    kkt_tol=1e-3,
    gradient_clip=10.0,
    print_every_outer=1,
    verbose=True,
):
    """
    Proper inequality Augmented-Lagrangian NOM.

    Core NOM logic is unchanged:
      x0 -> trainable start_layer -> x -> learned objective.

    What changes is ONLY the constraint treatment:
      - one lambda_i >= 0 per discretized inequality,
      - PHR augmented-Lagrangian loss,
      - multiplier updates between inner optimization blocks,
      - adaptive rho,
      - KKT diagnostics.

    Parameters are selected without using a known target objective.
    """
    n = problem["n_vars"]

    nom = NeuralOptimizationMachinePHR(
        obj_model,
        A,
        b,
        n_vars=n,
        rho=rho0,
    ).to(device)

    low, high = problem["train_interval"]

    torch.manual_seed(SEED)
    starting_points = (
        torch.rand(n_starting_points, n)
        * (high - low)
        + low
    )

    A_np = A.detach().cpu().numpy().astype(np.float64)
    b_np = b.detach().cpu().numpy().astype(np.float64)

    all_start_rows = []
    history_rows = []

    best_feasible = None
    best_kkt = None
    best_any = None

    started = time.perf_counter()

    for start_idx, x0 in enumerate(starting_points):
        x0 = x0.to(device).unsqueeze(0)

        # Preserve the original NOM generator idea, but reset it per start.
        nom.start_layer.reset_parameters()
        nom.reset_multipliers()
        nom.rho = float(rho0)

        optimizer = torch.optim.Adam(
            nom.start_layer.parameters(),
            lr=learning_rate,
        )

        previous_primal = float("inf")
        best_this_start = None
        best_kkt_this_start = None
        executed_outer = 0

        if verbose:
            print(
                f"\n--- PHR start {start_idx}: "
                f"{x0.squeeze().detach().cpu().numpy()} ---"
            )

        for outer in range(outer_iterations):
            executed_outer = outer + 1

            # ----------------------------
            # Inner optimization over NOM's start_layer
            # ----------------------------
            for inner in range(inner_epochs):
                optimizer.zero_grad()

                loss, x_current, g = nom(x0)
                loss.backward()

                if gradient_clip is not None:
                    torch.nn.utils.clip_grad_norm_(
                        nom.start_layer.parameters(),
                        max_norm=gradient_clip,
                    )

                optimizer.step()

                # Track the best feasible iterate using the TRUE objective.
                with torch.no_grad():
                    loss_eval, x_eval, g_eval = nom(x0)

                    x_np = (
                        x_eval.squeeze(0)
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(np.float64)
                    )

                    obj_original = original_objective_value(
                        x_np, problem
                    )
                    obj_transformed = transformed_objective_value(
                        x_np, problem
                    )

                    if g_eval.numel() == 0:
                        max_violation = 0.0
                    else:
                        max_violation = float(
                            torch.relu(g_eval)
                            .max()
                            .item()
                        )

                    candidate = {
                        "start": start_idx,
                        "outer": outer,
                        "inner": inner,
                        "epoch_equivalent": outer * inner_epochs + inner,
                        "x": x_np.copy(),
                        "objective_original": obj_original,
                        "objective_transformed": obj_transformed,
                        "max_violation": max_violation,
                        "nom_loss": float(loss_eval.item()),
                        "rho": float(nom.rho),
                        "lam": nom.lam.detach().cpu().numpy().astype(np.float64).copy(),
                    }

                    if (
                        best_any is None
                        or candidate["nom_loss"] < best_any["nom_loss"]
                    ):
                        best_any = candidate.copy()

                    if max_violation <= feasibility_tol:
                        if (
                            best_this_start is None
                            or obj_transformed
                            < best_this_start["objective_transformed"]
                        ):
                            best_this_start = candidate.copy()

                        if (
                            best_feasible is None
                            or obj_transformed
                            < best_feasible["objective_transformed"]
                        ):
                            best_feasible = candidate.copy()

            # ----------------------------
            # Multiplier update
            # ----------------------------
            with torch.no_grad():
                loss_eval, x_eval, g_eval = nom(x0)

                x_np = (
                    x_eval.squeeze(0)
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )

                if g_eval.numel() == 0:
                    primal_before_update = 0.0
                else:
                    primal_before_update = float(
                        torch.relu(g_eval).max().item()
                    )

                nom.update_multipliers(g_eval)

                lam_np = (
                    nom.lam.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )

                kkt = kkt_metrics_numpy(
                    problem,
                    x_np,
                    lam_np,
                    A_np,
                    b_np,
                )

                obj_original = original_objective_value(
                    x_np, problem
                )
                obj_transformed = transformed_objective_value(
                    x_np, problem
                )

                kkt_candidate = {
                    "start": start_idx,
                    "outer": outer,
                    "inner": inner_epochs - 1,
                    "epoch_equivalent": (outer + 1) * inner_epochs - 1,
                    "x": x_np.copy(),
                    "objective_original": obj_original,
                    "objective_transformed": obj_transformed,
                    "max_violation": kkt["primal_violation"],
                    "nom_loss": float(loss_eval.item()),
                    "rho": float(nom.rho),
                    "lam": lam_np.copy(),
                    **kkt,
                }

                if (
                    best_kkt_this_start is None
                    or kkt_candidate["kkt_merit"]
                    < best_kkt_this_start["kkt_merit"]
                ):
                    best_kkt_this_start = kkt_candidate.copy()

                if (
                    best_kkt is None
                    or kkt_candidate["kkt_merit"]
                    < best_kkt["kkt_merit"]
                ):
                    best_kkt = kkt_candidate.copy()

                history_rows.append(
                    {
                        "start": start_idx,
                        "outer": outer,
                        "objective_original": obj_original,
                        "objective_transformed": obj_transformed,
                        "primal_violation": kkt["primal_violation"],
                        "complementarity": kkt["complementarity"],
                        "stationarity": kkt["stationarity"],
                        "kkt_merit": kkt["kkt_merit"],
                        "rho": float(nom.rho),
                    }
                )

                # ----------------------------
                # Adaptive rho
                # ----------------------------
                # If feasibility is not improving sufficiently, strengthen rho.
                if (
                    kkt["primal_violation"] > feasibility_tol
                    and np.isfinite(previous_primal)
                    and kkt["primal_violation"]
                    > violation_reduction_target * previous_primal
                ):
                    nom.rho = min(
                        float(nom.rho) * rho_multiplier,
                        float(rho_max),
                    )

                previous_primal = kkt["primal_violation"]

            if verbose and (
                outer == 0
                or (outer + 1) % print_every_outer == 0
                or outer == outer_iterations - 1
            ):
                print(
                    f"  outer {outer:3d} | "
                    f"obj={obj_original:.8f} | "
                    f"primal={kkt['primal_violation']:.3e} | "
                    f"comp={kkt['complementarity']:.3e} | "
                    f"station={kkt['stationarity']:.3e} | "
                    f"KKT={kkt['kkt_merit']:.3e} | "
                    f"rho={nom.rho:.3g} | "
                    f"x={x_np}"
                )

            # KKT-based stop. No known answer is used.
            if (
                kkt["primal_violation"] <= feasibility_tol
                and kkt["complementarity"] <= kkt_tol
                and kkt["stationarity"] <= kkt_tol
            ):
                if verbose:
                    print(
                        f"  KKT stopping criterion reached "
                        f"at outer iteration {outer}."
                    )
                break

        # Choose the best feasible point from this start when available.
        chosen_for_start = (
            best_this_start
            if best_this_start is not None
            else best_kkt_this_start
        )

        if chosen_for_start is None:
            chosen_for_start = best_any

        row = {
            "start": start_idx,
            "outer_iterations_executed": executed_outer,
            "best_outer": int(chosen_for_start.get("outer", -1)),
            "best_inner": int(chosen_for_start.get("inner", -1)),
            "objective_original": float(chosen_for_start["objective_original"]),
            "objective_transformed": float(chosen_for_start["objective_transformed"]),
            "max_violation": float(chosen_for_start["max_violation"]),
            "rho": float(chosen_for_start["rho"]),
            "feasible": bool(
                chosen_for_start["max_violation"] <= feasibility_tol
            ),
        }

        kkt_for_row = kkt_metrics_numpy(
            problem,
            chosen_for_start["x"],
            chosen_for_start["lam"],
            A_np,
            b_np,
        )
        row.update(kkt_for_row)

        for j, value in enumerate(
            chosen_for_start["x"], start=1
        ):
            row[f"x{j}"] = float(value)

        all_start_rows.append(row)

    runtime = time.perf_counter() - started

    # Primary output: best feasible true-objective point.
    # Fallback: best KKT point, then best raw NOM loss.
    selected = (
        best_feasible
        if best_feasible is not None
        else best_kkt
        if best_kkt is not None
        else best_any
    )

    selected_kkt = kkt_metrics_numpy(
        problem,
        selected["x"],
        selected["lam"],
        A_np,
        b_np,
    )

    # best_kkt is an OUTER-iteration state measured immediately after
    # a multiplier update, so its KKT diagnostics are the most meaningful
    # convergence diagnostics. The selected point can still be the best
    # feasible objective point seen anywhere in the inner iterations.
    best_kkt_report = best_kkt if best_kkt is not None else selected
    best_kkt_metrics = kkt_metrics_numpy(
        problem,
        best_kkt_report["x"],
        best_kkt_report["lam"],
        A_np,
        b_np,
    )

    summary = {
        "problem_type": problem["problem_type"],
        "sense": problem["sense"],
        "n_vars": n,
        "penalty_mode": "phr_augmented_lagrangian",
        "rho0": float(rho0),
        "rho_selected": float(selected["rho"]),
        "rho_max": float(rho_max),
        "n_constraint_rows": int(A.shape[0]),
        "n_starting_points": int(n_starting_points),
        "outer_iterations": int(outer_iterations),
        "inner_epochs": int(inner_epochs),
        "total_possible_gradient_steps_per_start": int(
            outer_iterations * inner_epochs
        ),
        "runtime_sec": float(runtime),
        "selected_start": int(selected["start"]),
        "selected_outer": int(selected.get("outer", -1)),
        "selected_inner": int(selected.get("inner", -1)),
        "objective_original": float(selected["objective_original"]),
        "objective_transformed": float(selected["objective_transformed"]),
        "max_constraint_violation": float(selected["max_violation"]),
        "feasible": bool(
            selected["max_violation"] <= feasibility_tol
        ),

        # KKT values at the selected best-feasible objective point.
        "selected_primal_violation": selected_kkt["primal_violation"],
        "selected_complementarity": selected_kkt["complementarity"],
        "selected_stationarity": selected_kkt["stationarity"],
        "selected_kkt_merit": selected_kkt["kkt_merit"],

        # Best KKT state seen immediately after a multiplier update.
        "best_kkt_start": int(best_kkt_report["start"]),
        "best_kkt_outer": int(best_kkt_report.get("outer", -1)),
        "best_kkt_objective_original": float(
            best_kkt_report["objective_original"]
        ),
        "best_kkt_primal_violation": best_kkt_metrics["primal_violation"],
        "best_kkt_complementarity": best_kkt_metrics["complementarity"],
        "best_kkt_stationarity": best_kkt_metrics["stationarity"],
        "best_kkt_merit": best_kkt_metrics["kkt_merit"],
    }

    for j, value in enumerate(selected["x"], start=1):
        summary[f"x{j}"] = float(value)

    return (
        summary,
        pd.DataFrame(all_start_rows),
        selected,
        pd.DataFrame(history_rows),
    )


# ============================================================
# 8. DENSE VALIDATION FOR SEMI-INFINITE PROBLEMS
# ============================================================

def dense_semi_infinite_check(problem, x, n_dense=100_001):
    """
    Evaluate the ORIGINAL t-dependent constraint families on a dense t grid.
    This does not change NOM. It is only a post-solution validation step.
    """
    if problem["problem_type"] != "semi_infinite":
        return {
            "dense_max_violation": np.nan,
            "dense_worst_t": np.nan,
        }

    t = torch.linspace(
        float(problem["t_min"]),
        float(problem["t_max"]),
        int(n_dense),
        dtype=torch.float64,
    )

    x_t = torch.tensor(
        np.asarray(x, dtype=np.float64),
        dtype=torch.float64,
    )

    global_max = 0.0
    global_t = float(problem["t_min"])

    for family in problem["semi_families"]:
        cols = [
            evaluate_t_expression(expr, t)
            for expr in family["coeff_exprs"]
        ]
        A_dense = torch.column_stack(cols)
        b_dense = evaluate_t_expression(
            family["rhs_expr"], t
        )

        lhs = A_dense @ x_t

        if family["relation"] == "<=":
            violation = lhs - b_dense
        elif family["relation"] == ">=":
            violation = b_dense - lhs
        else:
            violation = torch.abs(lhs - b_dense)

        idx = int(torch.argmax(violation).item())
        value = float(max(0.0, violation[idx].item()))

        if value > global_max:
            global_max = value
            global_t = float(t[idx].item())

    return {
        "dense_max_violation": global_max,
        "dense_worst_t": global_t,
    }


# ============================================================
# 9. COMPLETE WORKFLOW
# ============================================================

def run_problem(
    problem,
    *,
    train_nsamples=10_000,
    test_nsamples=1_000,
    batch_size=64,
    surrogate_epochs=1_000,
    surrogate_lr=0.002,

    # Old penalty methods (kept for comparison)
    penalty=20.0,
    nom_epochs=10_000,

    # Proper PHR augmented-Lagrangian settings
    rho0=1.0,
    rho_max=1000.0,
    rho_multiplier=2.0,
    violation_reduction_target=0.7,
    outer_iterations=30,
    inner_epochs=300,
    kkt_tol=1e-3,
    gradient_clip=10.0,

    n_starting_points=5,
    nom_lr=0.001,
    feasibility_tol=1e-5,
    print_every=500,
    print_every_outer=1,
    save_results=True,
    output_dir="results",
    verbose=True,
):
    print("\nRunning on:", device)
    print("Problem type:", problem["problem_type"])
    print("Number of variables:", problem["n_vars"])
    print("Objective sense:", problem["sense"])
    print("Objective coefficients:", problem["objective_coeffs"])
    print("Constraint handling:", problem["penalty_mode"])

    A, b, constraint_metadata = build_constraint_matrix(problem)

    print("\nConstraint matrix created:")
    print("  A shape:", tuple(A.shape))
    print("  b shape:", tuple(b.shape))
    if problem["problem_type"] == "semi_infinite":
        print(
            f"  t interval=[{problem['t_min']}, {problem['t_max']}], "
            f"m={problem['m']}"
        )

    # ------------------------------------------------------------
    # Independent finite-LP benchmark
    # ------------------------------------------------------------
    benchmark = solve_finite_lp_benchmark(problem, A, b)

    if benchmark.get("lp_success", False):
        benchmark_x = np.array(
            [
                benchmark[f"lp_x{i+1}"]
                for i in range(problem["n_vars"])
            ]
        )
        print("\nIndependent finite-LP benchmark (SciPy/HiGHS):")
        print("  LP x =", benchmark_x)
        print(
            "  LP original objective =",
            benchmark["lp_objective_original"],
        )
        print(
            "  This benchmark is NOT used to train NOM; "
            "it is only a correctness check."
        )
    else:
        print(
            "\nLP benchmark unavailable:",
            benchmark.get("lp_message", "unknown reason"),
        )

    # ------------------------------------------------------------
    # Stage 1: objective surrogate
    # ------------------------------------------------------------
    obj_model, surrogate_test_mse = train_objective_surrogate(
        problem,
        train_nsamples=train_nsamples,
        test_nsamples=test_nsamples,
        batch_size=batch_size,
        epochs=surrogate_epochs,
        learning_rate=surrogate_lr,
        verbose=verbose,
    )

    history_df = pd.DataFrame()

    # ------------------------------------------------------------
    # Stage 2: NOM
    # ------------------------------------------------------------
    if problem["penalty_mode"] == "phr_augmented_lagrangian":
        (
            summary,
            all_starts_df,
            selected,
            history_df,
        ) = optimize_with_nom_phr(
            problem,
            obj_model,
            A,
            b,
            rho0=rho0,
            rho_max=rho_max,
            rho_multiplier=rho_multiplier,
            violation_reduction_target=violation_reduction_target,
            n_starting_points=n_starting_points,
            outer_iterations=outer_iterations,
            inner_epochs=inner_epochs,
            learning_rate=nom_lr,
            feasibility_tol=feasibility_tol,
            kkt_tol=kkt_tol,
            gradient_clip=gradient_clip,
            print_every_outer=print_every_outer,
            verbose=verbose,
        )
    else:
        # Original standard/quadratic-augmented penalty path.
        summary, all_starts_df, selected = optimize_with_nom(
            problem,
            obj_model,
            A,
            b,
            penalty=penalty,
            n_starting_points=n_starting_points,
            n_epochs_per_start=nom_epochs,
            learning_rate=nom_lr,
            feasibility_tol=feasibility_tol,
            print_every=print_every,
            verbose=verbose,
        )

        # Add KKT-like fields as NaN because old penalty methods
        # do not maintain proper Lagrange multipliers.
        summary["primal_violation"] = summary["max_constraint_violation"]
        summary["complementarity"] = np.nan
        summary["stationarity"] = np.nan
        summary["kkt_merit"] = np.nan

    summary["surrogate_test_mse"] = float(surrogate_test_mse)

    # ------------------------------------------------------------
    # Dense validation of the ORIGINAL semi-infinite constraint
    # ------------------------------------------------------------
    dense = dense_semi_infinite_check(
        problem,
        selected["x"],
    )
    summary.update(dense)

    # ------------------------------------------------------------
    # Benchmark comparison
    # ------------------------------------------------------------
    summary.update(benchmark)

    if benchmark.get("lp_success", False):
        summary["objective_gap_vs_lp"] = abs(
            float(summary["objective_original"])
            - float(benchmark["lp_objective_original"])
        )

        nom_x = np.array(
            [
                summary[f"x{i+1}"]
                for i in range(problem["n_vars"])
            ],
            dtype=np.float64,
        )
        lp_x = np.array(
            [
                benchmark[f"lp_x{i+1}"]
                for i in range(problem["n_vars"])
            ],
            dtype=np.float64,
        )
        summary["x_distance_vs_lp"] = float(
            np.linalg.norm(nom_x - lp_x)
        )
    else:
        summary["objective_gap_vs_lp"] = np.nan
        summary["x_distance_vs_lp"] = np.nan

    print("\n" + "=" * 72)
    print("FINAL NOM RESULT")
    print("=" * 72)

    final_x = np.array(
        [
            summary[f"x{i+1}"]
            for i in range(problem["n_vars"])
        ]
    )

    print("x =", final_x)
    print(
        "Original objective value:",
        summary["objective_original"],
    )
    print(
        "Transformed minimization objective:",
        summary["objective_transformed"],
    )
    print(
        "Maximum discretized constraint violation:",
        summary["max_constraint_violation"],
    )

    if problem["penalty_mode"] == "phr_augmented_lagrangian":
        print("\nKKT diagnostics (best multiplier-updated state):")
        print("  primal violation :", summary["best_kkt_primal_violation"])
        print("  complementarity  :", summary["best_kkt_complementarity"])
        print("  stationarity      :", summary["best_kkt_stationarity"])
        print("  KKT merit         :", summary["best_kkt_merit"])
        print("  selected rho      :", summary["rho_selected"])

    if problem["problem_type"] == "semi_infinite":
        print(
            "\nDense original-LSIP violation:",
            summary["dense_max_violation"],
        )
        print(
            "Worst dense t:",
            summary["dense_worst_t"],
        )

    print(
        "Feasible on discretized constraints:",
        summary["feasible"],
    )
    print("Selected start:", summary["selected_start"])
    print("Runtime (s):", summary["runtime_sec"])

    if benchmark.get("lp_success", False):
        print("\nNOM vs finite-LP benchmark:")
        print(
            "  objective gap =",
            summary["objective_gap_vs_lp"],
        )
        print(
            "  x-distance    =",
            summary["x_distance_vs_lp"],
        )

    # ------------------------------------------------------------
    # Save experiment artifacts
    # ------------------------------------------------------------
    if save_results:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)

        summary_path = out / "generic_nom_summary.csv"
        starts_path = out / "generic_nom_all_starts.csv"
        history_path = out / "generic_nom_phr_history.csv"

        pd.DataFrame([summary]).to_csv(
            summary_path,
            index=False,
        )
        all_starts_df.to_csv(
            starts_path,
            index=False,
        )

        if not history_df.empty:
            history_df.to_csv(
                history_path,
                index=False,
            )

        print("\nSaved:")
        print(summary_path.resolve())
        print(starts_path.resolve())
        if not history_df.empty:
            print(history_path.resolve())

    return {
        "problem": problem,
        "A": A,
        "b": b,
        "constraint_metadata": constraint_metadata,
        "objective_model": obj_model,
        "summary": summary,
        "all_starts": all_starts_df,
        "phr_history": history_df,
        "benchmark": benchmark,
    }


# ============================================================
# 10. INTERACTIVE ENTRY POINT
# ============================================================

## 2. Enter the optimization problem

Choose `1` for a finite linear program or `2` for a linear semi-infinite program.

For LSIP coefficient functions, accepted examples include `1`, `t`, `t**2`, `t+t**2`, `exp(t)`, `sin(t)`, `sqrt(1+t)`.

In [4]:
problem = collect_problem_from_user()
problem


INTERACTIVE NEURAL OPTIMIZATION MACHINE (NOM)
Choose the type of problem:
  1. Linear programming (finite linear constraints)
  2. Semi-linear / Linear Semi-Infinite Programming (LSIP)
     Finite x variables, but constraints indexed by t in an interval.

Semi-infinite parameter interval

Allowed t expressions include:
  1, t, t**2, t+t**2, exp(t), sin(t), cos(t), sqrt(1+t)
Each constraint must stay LINEAR in x; only its coefficients may depend on t.

Semi-infinite constraint family 1

Neural surrogate training interval

Constraint-handling method:
  1. Standard penalty: rho * sum(violation)
  2. Quadratic-augmented penalty: rho * sum(v + 0.5*v^2)
  3. Proper inequality Augmented Lagrangian (recommended)
     Uses one multiplier per constraint, adaptive rho, and KKT checks.


{'problem_type': 'semi_infinite',
 'n_vars': 3,
 'sense': 'max',
 'objective_coeffs': array([1.        , 0.5       , 0.83333333]),
 'nonnegative': True,
 'finite_constraints': [],
 'semi_families': [{'coeff_exprs': ['1', 't', 't + t**2'],
   'relation': '<=',
   'rhs_expr': 'exp(t)',
   'name': 'semi_family_1'}],
 't_min': 0.0,
 't_max': 1.0,
 'm': 1001,
 'train_interval': (0.0, 2.0),
 'penalty_mode': 'phr_augmented_lagrangian'}

## 3. Run NOM

If you select method **3**, these settings use the proper PHR augmented Lagrangian. The known solution is never used by the optimizer.

In [5]:
results = run_problem(
    problem,
    train_nsamples=10_000,
    test_nsamples=1_000,
    batch_size=64,
    surrogate_epochs=1_000,
    surrogate_lr=0.002,

    # Old penalty methods, retained for comparison
    penalty=20.0,
    nom_epochs=10_000,

    # Proper PHR augmented-Lagrangian settings
    rho0=1.0,
    rho_max=1000.0,
    rho_multiplier=2.0,
    violation_reduction_target=0.7,
    outer_iterations=30,
    inner_epochs=300,
    kkt_tol=1e-3,
    gradient_clip=10.0,

    n_starting_points=5,
    nom_lr=0.001,
    feasibility_tol=1e-5,
    print_every=500,
    print_every_outer=1,
    save_results=True,
    output_dir='results',
    verbose=True,
)


Running on: cuda
Problem type: semi_infinite
Number of variables: 3
Objective sense: max
Objective coefficients: [1.         0.5        0.83333333]
Constraint handling: phr_augmented_lagrangian

Constraint matrix created:
  A shape: (1004, 3)
  b shape: (1004,)
  t interval=[0.0, 1.0], m=1001

Independent finite-LP benchmark (SciPy/HiGHS):
  LP x = [1.         0.10691846 0.7888096 ]
  LP original objective = 1.7108005617491981
  This benchmark is NOT used to train NOM; it is only a correctness check.

Training the objective surrogate...
  epoch    0 | train MSE=0.86868042 | validation MSE=0.00102714
  epoch  500 | train MSE=0.00000037 | validation MSE=0.00000063
  epoch  999 | train MSE=0.00000108 | validation MSE=0.00000028
Final objective-surrogate test MSE: 0.0000002785

--- PHR start 0: [1.7645385 1.8300079 0.7657275] ---
  outer   0 | obj=1.57725528 | primal=8.713e-01 | comp=7.592e-01 | station=7.140e-01 | KKT=2.344e+00 | rho=1 | x=[ 0.86786294 -0.87132114  1.37406349]
  outer   

## Example: the paper's printed 3-variable problem with ε = 1

Use:

- Problem type: `2`
- Variables: `3`
- Sense: `max`
- Objective coefficients: `1 0.5 0.8333333333333333`
- Nonnegative: `y`
- `t_min = 0`, `t_max = 1`
- `m = 101` or `1001`
- Semi-infinite families: `1`
- x1 coefficient: `1`
- x2 coefficient: `t`
- x3 coefficient: `t + t**2`
- Relation: `<=`
- RHS: `exp(t)`
- Additional fixed constraints: `0`
- Surrogate interval: `0` to `2`
- Constraint method: `3`

**Important validation note:** for the mathematical problem exactly as printed with `t in [0,1]`, the independent discretized LP benchmark is about **1.7108** on a fine grid. The notebook prints this benchmark automatically so NOM can be judged against the actual entered problem rather than forced toward a known target.